## Example of kernel and Gaussian process simulations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve
from scipy.stats import multivariate_normal

# --- 1. Kernel definition ---
def expKern(x, y, param):
    sigma, theta = param
    dist = np.subtract.outer(x / theta, y / theta)
    kern = (sigma ** 2) * np.exp(- np.abs(dist))
    return kern

def mat5_2Kern(x, y, param):
    sigma, theta = param
    d = np.abs(np.subtract.outer(x, y)) * np.sqrt(5) / theta
    kern = (sigma ** 2) * (1 + d + d**2 / 3) * np.exp(-d)
    return kern

# --- 2. Simulation of sample paths of the associated centered Gaussian process ---
kern = mat5_2Kern
param = np.array([1, 0.2])  # covariance parameters

# 1. Simulation of sample paths
n = 250
x = np.linspace(0, 1, n)

# Computation of the covariance matrix k(x_i, x_j)
kxx = kern(x, x, param)

# Image plot 
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(kxx, cmap='viridis', origin='lower', extent=[0, 1, 0, 1])
plt.title("kernel plot")
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar()

# Simulations
samples = np.random.multivariate_normal(mean=np.zeros(n), cov=kxx, size=5).T
plt.subplot(1, 2, 2)
plt.plot(x, samples)
plt.title("sample paths")
plt.show()

Q: Explain the form of the image plot of the kernel. 

Q: Play with the lengthscale parameter (theta). Interpretation?

## Gaussian process regression

Let us now use Gaussian processes to interpolate a toy function.

In [ ]:
# --- 3. Gaussian process regression ---
# Function, learning set or 'design of experiments'  
def fun(x):
    return x + np.sin(4 * np.pi * x)

X = np.linspace(0.1, 1, 6)
Y = fun(X)

# Plot
plt.figure(figsize=(8, 6))
x_plot = np.linspace(0, 1, n)
plt.plot(x_plot, fun(x_plot), linestyle='dotted', label='function f(x)')
plt.scatter(X, Y, color='black', s=100, label='training points')
plt.ylim(-1.5, 4)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()



In [ ]:
# --- 4. Conditional mean and kernel ---
def condMean(x, X, Y, kern, param, jitter=0):
    K = kern(X, X, param)
    Kinv = solve(K + jitter * np.eye(len(K)), np.eye(len(K)))
    kxX = kern(x, X, param)
    return kxX @ Kinv @ Y

def condCov(x, X, kern, param, jitter=0):
    K = kern(X, X, param)
    Kinv = solve(K + jitter * np.eye(len(K)), np.eye(len(K)))
    kxX = kern(x, X, param)
    kxx = kern(x, x, param)
    return kxx - kxX @ Kinv @ kxX.T

kern = mat5_2Kern
param = np.array([1, 0.2])

mu = condMean(x_plot, X, Y, kern, param, jitter=0)
Sigma = condCov(x_plot, X, kern, param, jitter=0)
varSigma = np.diag(Sigma)
varSigma = np.maximum(varSigma, 0)  # to avoid numerical negative values

plt.figure(figsize=(8, 6))
plt.plot(x_plot, fun(x_plot), linestyle='dotted', label='function f(x)')
plt.scatter(X, Y, color='black', s=100, label='training points')
plt.plot(x_plot, mu, color='blue', linewidth=3, label='cond. mean m(x)')
plt.plot(x_plot, mu + 1.96 * np.sqrt(varSigma), linestyle="dashed", color = "blue", label='95% pred. inter')
plt.plot(x_plot, mu - 1.96 * np.sqrt(varSigma), linestyle="dashed", color = "blue")
plt.legend()
plt.ylim(-1.5, 4)
plt.xlabel("x")
plt.ylabel("y")
plt.show()


In [ ]:
# --- 5. Conditional simulations ---
np.random.seed(42)
nSim=10
condSamples = np.random.multivariate_normal(mean=mu, cov=Sigma, size=nSim).T
plt.figure(figsize=(8, 6))
plt.plot(x_plot, condSamples, color='grey', alpha=0.3)
plt.scatter(X, Y, color='black', s=100, label='training points')
plt.plot(x_plot, fun(x_plot), color='black', linewidth=2, label='function f(x)')
plt.legend()
plt.ylim(-1.5, 4)
plt.xlabel("x")
plt.ylabel("y")
plt.show()


In [ ]:
# Link with conditional mean and prediction intervals
plt.figure(figsize=(8, 6))
plt.plot(x_plot, fun(x_plot), linestyle='dotted', label='function f(x)')
plt.scatter(X, Y, color='black', s=100, label='training points')
plt.plot(x_plot, condSamples, color='grey', alpha=0.3)
plt.plot(x_plot, np.mean(condSamples, axis=1), color='blue', linewidth=4, label='empirical mean (or c.i.)')
plt.plot(x_plot, mu, color='red', linestyle='dotted', linewidth=4, label='theoretical mean (or c.i.)')
plt.plot(x_plot, np.quantile(condSamples, 0.025, axis=1), color='blue', linewidth=2)
plt.plot(x_plot, np.quantile(condSamples, 0.975, axis=1), color='blue', linewidth=2)
plt.plot(x_plot, mu + 1.96 * np.sqrt(varSigma), color='red', linestyle='dotted', linewidth=2)
plt.plot(x_plot, mu - 1.96 * np.sqrt(varSigma), color='red', linestyle='dotted', linewidth=2)
plt.legend()
plt.ylim(-1.5, 4)
plt.xlabel("x")
plt.ylabel("y")
plt.show()


Q: What do you observe when nSim increases (say up to 1000)?

## Gaussian process regression with a symmetry information

Now, we consider the same function on the double sized interval [-1, 1].
We want to compare a non-informed kernel with a kernel containing the information that the function is odd: f(x) = -f(-x).
Here we choose to keep the same design of experiments defined on only half of the domain [0, 1] with the same size.

In [ ]:
# --- 6. GP regression with/without a symmetry information ---
def kernSymm(x, y, param):  # to be corrected!!
    return (mat5_2Kern(x, y, param))

kern = kernSymm  # try mat5_2Kern and kernSymm.
x_symm = np.linspace(-1, 1, 250)

mu_symm = condMean(x_symm, X, Y, kern, param, jitter=0)
Sigma_symm = condCov(x_symm, X, kern, param, jitter=0)
varSigma_symm = np.diag(Sigma_symm)
varSigma_symm = np.maximum(varSigma_symm, 0)

plt.figure(figsize=(8, 6))
plt.plot(x_symm, fun(x_symm), label='function f(x)')
plt.scatter(X, Y, color='black', s=100, label='training points')
plt.plot(x_symm, mu_symm, color='blue', label='cond. mean m(x)')
plt.plot(x_symm, mu_symm + 1.96 * np.sqrt(varSigma_symm), color='gray', linestyle='--', label='95% pred. inter')
plt.plot(x_symm, mu_symm - 1.96 * np.sqrt(varSigma_symm), color='gray', linestyle='--')
plt.legend()
plt.ylim(-2, 2)
plt.xlim(-1, 1)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# Échantillons conditionnels
condSamples_symm = np.random.multivariate_normal(mean=mu_symm, cov=Sigma_symm, size=2).T
plt.figure(figsize=(8, 6))
plt.plot(x_symm, condSamples_symm, color='grey', linewidth=1)
plt.scatter(X, Y, color='black', s=100, label='training points')
plt.plot(x_symm, fun(x_symm), color='black', linewidth=1, label='function f(x)')
plt.legend()
plt.ylim(-2, 2)
plt.xlim(-1, 1)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

Q: What's happening with the Matérn kernel when extrapolating? (i.e. predicting at a point located 'far' from the design points?) Can you explain it?

Q: What do you observe with the symmetric kernel? (mean, s.d. and sample paths)